# 第 3 章 · 一轮 step 的完整生命线

**这一章你会得到什么**：看清一次 `step()` 内部 Model、Environment、observation formatter 的**真实调用顺序**，并认全 `system / user / assistant / tool` 四种消息 role。

## 📖 对照源码（在 IDE 里打开这些文件，边看边跑）

- `src/minisweagent/agents/default.py` **L124–126** — `step()`
- `src/minisweagent/agents/default.py` **L128–150** — `query()`（调 model.query + 记账 + 追加消息）
- `src/minisweagent/agents/default.py` **L152–155** — `execute_actions()`（调 env.execute + 追加 observation）
- `src/minisweagent/models/test_models.py` **L177–188** — `format_observation_messages`（结果变消息）

> 快捷：代码格里 `函数名??` 直接打印源码；或用 `show_source("相对路径", 起始行, 结束行)`。

In [1]:
# 环境自检：把源码目录加入 sys.path，并切到仓库根目录
import os, sys
from pathlib import Path
os.environ["MSWEA_SILENT_STARTUP"] = "1"
REPO = Path(r"/Users/xinranzhao/Documents/llm-study/books/mini-swe-agent-source-guide/mini-swe-agent")
SRC = REPO / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
os.chdir(REPO)
import minisweagent
print("Python:", sys.version.split()[0])
print("mini-SWE-agent:", minisweagent.__version__)
print("仓库根目录:", REPO)

Python: 3.13.14
mini-SWE-agent: 2.4.5
仓库根目录: /Users/xinranzhao/Documents/llm-study/books/mini-swe-agent-source-guide/mini-swe-agent


## 概念：一次 step 拆成两半

```python
def step(self):
    return self.execute_actions(self.query())
```
- `query()`：检查限制 -> 调 `model.query()` -> 累加 cost -> 把 assistant 消息追加进历史
- `execute_actions()`：对每个 action 调 `env.execute()` -> 把结果交给 `model.format_observation_messages()` 变成 `tool` 消息 -> 追加进历史

In [2]:
# 小工具：带行号打印源码切片（相当于 nl + sed）
def show_source(rel_path: str, start: int, end: int) -> None:
    lines = (REPO / rel_path).read_text().splitlines()
    end = min(end, len(lines))
    w = len(str(end))
    for n in range(start, end + 1):
        print(f"{n:>{w}}  {lines[n - 1]}")

In [3]:
show_source("src/minisweagent/agents/default.py", 124, 155)

124      def step(self) -> list[dict]:
125          """Query the LM, execute actions."""
126          return self.execute_actions(self.query())
127  
128      def query(self) -> dict:
129          """Query the model and return model messages. Override to add hooks."""
130          if 0 < self.config.step_limit <= self.n_calls or 0 < self.config.cost_limit <= self.cost:
131              raise LimitsExceeded(
132                  {
133                      "role": "exit",
134                      "content": "LimitsExceeded",
135                      "extra": {"exit_status": "LimitsExceeded", "submission": ""},
136                  }
137              )
138          if 0 < self.config.wall_time_limit_seconds <= int(time.time() - self._start_time):
139              raise TimeExceeded(
140                  {
141                      "role": "exit",
142                      "content": "TimeExceeded",
143                      "extra": {"exit_status": "TimeExceeded", "submission": ""},
144     

## 概念：四种 role 各自的来源

- `system`：`run()` 开头由 `system_template` 渲染，只出现一次，永不修改
- `user`：任务描述（`instance_template`），以及格式出错时的纠错反馈
- `assistant`：模型返回的原始消息（含 `tool_calls` 和解析出的 `extra.actions`）
- `tool`：命令执行结果，带 `tool_call_id` 与对应的 assistant tool call 配对

## 动手：给 Model 和 Environment 各套一层“追踪外壳”

补全两个包装类的 `execute` / `query`，让它们在真正调用前，往 `events` 里记一笔。
这利用的正是第 2 章的鸭子类型——包装类只要提供相同方法即可。

In [4]:
events = []

class TraceEnv:
    def __init__(self, inner):
        self.inner = inner
    def execute(self, action, cwd=""):
        # TODO: 往 events 追加一条 f"Environment.execute({action['command']!r})"
        # TODO: 然后 return self.inner.execute(action, cwd)
        events.append(f"Environment.execute({action['command']!r})")
        return self.inner.execute(action, cwd)
    def get_template_vars(self, **kwargs):
        return self.inner.get_template_vars(**kwargs)
    def serialize(self):
        return self.inner.serialize()

## 运行看结果：完整事件顺序（参考实现）

In [5]:
from minisweagent.agents.default import DefaultAgent
from minisweagent.environments.local import LocalEnvironment
from minisweagent.models.test_models import DeterministicToolcallModel, make_toolcall_output

events = []

class TraceModel:
    def __init__(self, inner): self.inner = inner
    def query(self, messages, **kw):
        events.append(f"Model.query(messages={len(messages)})")
        return self.inner.query(messages, **kw)
    def format_message(self, **kw): return self.inner.format_message(**kw)
    def format_observation_messages(self, message, outputs, template_vars=None):
        events.append(f"Model.format_observation_messages(outputs={len(outputs)})")
        return self.inner.format_observation_messages(message, outputs, template_vars)
    def get_template_vars(self, **kw): return self.inner.get_template_vars(**kw)
    def serialize(self): return self.inner.serialize()

class TraceEnvRef:
    def __init__(self, inner): self.inner = inner
    def execute(self, action, cwd=""):
        events.append(f"Environment.execute({action['command']!r})")
        return self.inner.execute(action, cwd)
    def get_template_vars(self, **kw): return self.inner.get_template_vars(**kw)
    def serialize(self): return self.inner.serialize()

insp = {"command": "printf observation", "tool_call_id": "call_1"}
submit = {"command": "echo COMPLETE_TASK_AND_SUBMIT_FINAL_OUTPUT\necho traced", "tool_call_id": "call_2"}
inner = DeterministicToolcallModel(outputs=[
    make_toolcall_output("执行动作", [], [insp]),
    make_toolcall_output("提交", [], [submit]),
])
agent = DefaultAgent(TraceModel(inner), TraceEnvRef(LocalEnvironment(cwd=str(REPO))),
                     system_template="system", instance_template="{{task}}", cost_limit=5)
result = agent.run("追踪一次完整生命线")
print(*events, sep="\n")
print("\nresult =", result)

Model.query(messages=2)
Environment.execute('printf observation')
Model.format_observation_messages(outputs=1)
Model.query(messages=4)
Environment.execute('echo COMPLETE_TASK_AND_SUBMIT_FINAL_OUTPUT\necho traced')

result = {'exit_status': 'Submitted', 'submission': 'traced\n'}


In [7]:
print("消息轨迹：")
for i, m in enumerate(agent.messages):
    print(i, m.get("role"), repr(m.get("content"))[:70])

消息轨迹：
0 system 'system'
1 user '追踪一次完整生命线'
2 assistant '执行动作'
3 tool '<returncode>0</returncode>\n<output>\nobservation</output>'
4 assistant '提交'
5 exit 'traced\n'


## 观察点

- 事件顺序印证了 `query -> execute` 的节拍：每一步都是先问模型、再执行、再把 observation 格式化回去。
- 最后一条消息 role 是 `exit`（不是标准 OpenAI role），由 Environment 检测到完成信号后产生——下一章展开。
- `format_observation_messages` 由 **Model** 提供而不是 Environment：因为“结果怎么变成模型能读的消息”属于模型接口的职责。

## 闭卷检查
1. `query()` 返回什么？为什么还要 `add_messages()`？
2. `tool` 消息里的 `tool_call_id` 有什么用？
3. 为什么 observation 的格式化放在 Model 而不是 Environment？